# Mushroom Observation Data Pipeline - Kaggle Notebook

This notebook runs the environmental data enrichment pipeline for mushroom observations from iNaturalist.

## Setup Instructions

### Before Running:
1. **Enable Internet** in Kaggle notebook settings (Settings → Internet → On)
2. **Add Secrets** (Kaggle Add-ons → Secrets):
   - `OPENTOPOGRAPHY_API_KEY`: Your OpenTopography API key
   - `EARTHENGINE_PROJECT`: Your Google Cloud project ID (for Earth Engine)
   - Optional: `CDSAPI_URL` and `CDSAPI_KEY` for Copernicus CDS soil moisture
3. **Mount Google Drive** (optional): For NDVI exports from Earth Engine

### Data Storage:
Kaggle provides `/kaggle/working/` for output files that persist after the session.

## Step 1: Install Dependencies

In [ ]:
!pip install -q pyinaturalist meteostat earthengine-api cdsapi requests rasterio rio-cogeo xarray netCDF4 numpy scipy pandas scikit-learn

## Step 2: Configure Environment

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

# Set up working directory for Kaggle
WORKING_DIR = '/kaggle/working'
DATA_DIR = os.path.join(WORKING_DIR, 'data')
os.environ['DATA_DIR'] = DATA_DIR

# Load secrets from Kaggle Add-ons
user_secrets = UserSecretsClient()

# Required: OpenTopography API key for DEM data
try:
    opentopography_key = user_secrets.get_secret('OPENTOPOGRAPHY_API_KEY')
    os.environ['OPENTOPOGRAPHY_API_KEY'] = opentopography_key
    print("✓ OpenTopography API key loaded")
except Exception as e:
    print(f"⚠ OpenTopography API key not found: {e}")
    print("  Add it in Add-ons → Secrets → OPENTOPOGRAPHY_API_KEY")

# Required: Google Cloud project for Earth Engine
try:
    ee_project = user_secrets.get_secret('EARTHENGINE_PROJECT')
    os.environ['EARTHENGINE_PROJECT'] = ee_project
    print("✓ Earth Engine project ID loaded")
except Exception as e:
    print(f"⚠ Earth Engine project not found: {e}")
    print("  Add it in Add-ons → Secrets → EARTHENGINE_PROJECT")

# Optional: Copernicus CDS credentials for soil moisture
try:
    cds_url = user_secrets.get_secret('CDSAPI_URL')
    cds_key = user_secrets.get_secret('CDSAPI_KEY')
    os.environ['CDSAPI_URL'] = cds_url
    os.environ['CDSAPI_KEY'] = cds_key
    # Create .cdsapirc file
    with open(os.path.expanduser('~/.cdsapirc'), 'w') as f:
        f.write(f'url: {cds_url}\nkey: {cds_key}\n')
    print("✓ CDS API credentials loaded")
except Exception as e:
    print(f"⚠ CDS API credentials not found: {e}")
    print("  Soil moisture data will be skipped (optional)")

# Create necessary directories
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, 'species'), exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, 'enriched'), exist_ok=True)
os.makedirs(os.path.join(WORKING_DIR, 'dem'), exist_ok=True)
os.makedirs(os.path.join(WORKING_DIR, 'ndvi'), exist_ok=True)
os.makedirs(os.path.join(WORKING_DIR, 'soil'), exist_ok=True)

print(f"\nData directory: {DATA_DIR}")
print(f"Working directory: {WORKING_DIR}")

## Step 3: Initialize Earth Engine

In [ ]:
import ee

# Authenticate and initialize Earth Engine
# In Kaggle, this uses service account or OAuth based on your setup
try:
    ee.Initialize(project=os.environ.get('EARTHENGINE_PROJECT'))
    print("✓ Earth Engine initialized successfully")
except Exception as e:
    print(f"⚠ Earth Engine initialization failed: {e}")
    print("  Try running: ee.Authenticate() first, then ee.Initialize()")
    # Fallback attempt
    try:
        ee.Initialize()
        print("✓ Earth Engine initialized with default credentials")
    except Exception as e2:
        print(f"⚠ Earth Engine not available: {e2}")
        print("  NDVI and satellite moisture layers will be skipped")

## Step 4: Run the Pipeline

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working')

# Import pipeline modules
import species_store as store
import fetch
import terrain_pipeline as terrain
import enrich_with_rasters as enrich
import cluster
import export_geojson

print("Pipeline modules loaded successfully")

### 4.1: Fetch iNaturalist Observations

In [ ]:
# Run iNaturalist fetch
# This pulls mushroom observations and saves them to data/species/

print("Fetching iNaturalist observations...")
%run /kaggle/working/iNat.py

# Check what was fetched
species_files = list(store.species_slugs(store.SPECIES_DIR))
print(f"\n✓ Fetched {len(species_files)} species")
if species_files:
    print(f"  Species: {', '.join(species_files[:5])}{'...' if len(species_files) > 5 else ''}")

### 4.2: Download Environmental Layers

In [ ]:
# Download all environmental data
# - NDVI (Sentinel-2 via Earth Engine)
# - Soil moisture (ERA5-Land)
# - Precipitation (CHIRPS)
# - Land cover (ESA WorldCover)
# - Topography (SRTM DEM)

print("Downloading environmental layers...")
%run /kaggle/working/fetch.py

print("\n✓ Environmental layers downloaded")

### 4.3: Process Terrain DEM

In [ ]:
# Process DEM into terrain exposure layers
# Creates: slope, aspect, solar_exposure, wind_exposure, water_retention

dem_path = f"{WORKING_DIR}/dem/dem_SRTMGL3.tif"
if os.path.exists(dem_path):
    print(f"Processing DEM: {dem_path}")
    terrain.process_dem(dem_path)
    print("\n✓ Terrain layers derived")
else:
    print(f"⚠ DEM not found at {dem_path}")
    print("  Ensure fetch.py completed successfully")

### 4.4: Enrich Observations with Raster Data

In [ ]:
# Sample rasters at each observation point
# Output: data/enriched/<species>.csv with all environmental columns

print("Enriching observations with raster data...")
%run /kaggle/working/enrich_with_rasters.py

enriched_files = list(store.enriched_slugs(store.ENRICHED_DIR))
print(f"\n✓ Enriched {len(enriched_files)} species")

### 4.5: Cluster Observations

In [ ]:
# KMeans clustering by environmental similarity
# Adds 'cluster' column to enriched CSVs

print("Clustering observations by environmental similarity...")
%run /kaggle/working/cluster.py

print("\n✓ Clustering complete")

### 4.6: Export GeoJSON for Visualization

In [ ]:
# Export to GeoJSON format for mapping
# Output: public/data/observations.geojson (or working directory equivalent)

print("Exporting GeoJSON...")
%run /kaggle/working/export_geojson.py

# Verify output
geojson_path = f"{WORKING_DIR}/observations.geojson"
if os.path.exists(geojson_path):
    size_mb = os.path.getsize(geojson_path) / (1024 * 1024)
    print(f"\n✓ Exported: {geojson_path} ({size_mb:.2f} MB)")
else:
    # Check alternative location
    alt_path = f"{WORKING_DIR}/public/data/observations.geojson"
    if os.path.exists(alt_path):
        size_mb = os.path.getsize(alt_path) / (1024 * 1024)
        print(f"\n✓ Exported: {alt_path} ({size_mb:.2f} MB)")

## Step 5: Review Results

In [ ]:
import pandas as pd
from pathlib import Path

# List all output files
print("=== Output Files ===")
for root, dirs, files in os.walk(WORKING_DIR):
    # Skip hidden directories and __pycache__
    dirs[:] = [d for d in dirs if not d.startswith('.') and d != '__pycache__']
    level = root.replace(WORKING_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files[:10]:  # Limit to first 10 files per directory
        filepath = os.path.join(root, file)
        size = os.path.getsize(filepath)
        if size > 1024 * 1024:
            size_str = f"{size / (1024*1024):.1f}MB"
        elif size > 1024:
            size_str = f"{size / 1024:.1f}KB"
        else:
            size_str = f"{size}B"
        print(f'{subindent}{file} ({size_str})')
    if len(files) > 10:
        print(f'{subindent}... and {len(files) - 10} more files')

In [ ]:
# Preview enriched data for first species
enriched_dir = Path(DATA_DIR) / 'enriched'
enriched_files = list(enriched_dir.glob('*.csv'))

if enriched_files:
    first_file = enriched_files[0]
    print(f"Preview: {first_file.name}")
    df = pd.read_csv(first_file, nrows=5)
    
    # Show key columns
    display_cols = [col for col in df.columns if col in [
        'uuid', 'species_name', 'latitude', 'longitude', 
        'observed_on', 'elevation', 'cluster',
        'ndvi', 'soil_moisture', 'precip_7day',
        'slope', 'aspect', 'solar_exposure', 
        'wind_exposure', 'water_retention'
    ]]
    
    if display_cols:
        display(df[display_cols])
    else:
        display(df.head())
else:
    print("No enriched files found yet")

## Step 6: Download Results (Optional)

In [ ]:
# Create a zip archive of all results for download
import shutil
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
archive_name = f'mushroom_data_{timestamp}'
archive_path = shutil.make_archive(
    os.path.join(WORKING_DIR, archive_name),
    'zip',
    WORKING_DIR,
    base_dir='.'
)

print(f"✓ Created archive: {archive_path}")
print(f"  Size: {os.path.getsize(archive_path) / (1024*1024):.1f} MB")
print("\nTo download:")
print("1. Click on the 'Output' tab in Kaggle")
print(f"2. Find and download {archive_name}.zip")
print("   OR use the file browser on the right panel")

## Notes & Troubleshooting

### Common Issues:

1. **Earth Engine Authentication**: If EE initialization fails:
   ```python
   ee.Authenticate()
   ee.Initialize(project='your-project-id')
   ```

2. **NDVI Exports**: Earth Engine exports NDVI to Google Drive. You'll need to:
   - Monitor the export in Google Earth Engine Tasks
   - Download the GeoTIFF manually to `/kaggle/working/ndvi/`
   - Re-run the enrichment step

3. **Memory Limits**: Kaggle notebooks have ~16GB RAM. If you hit limits:
   - Reduce the number of species being processed
   - Use `REFRESH_ALL=0` to skip re-fetching existing data
   - Process species in batches

4. **Session Timeouts**: Kaggle sessions timeout after ~12 hours. Save outputs frequently:
   ```python
   # Commit intermediate results
   %run /kaggle/working/export_geojson.py
   ```

### Resuming Interrupted Runs:

The pipeline is designed to be resumable. Simply re-run from the failed step:
- Re-run cell 4.2 if downloads failed
- Re-run cell 4.4 if enrichment failed (it skips completed files)
- The `.done` marker in `data/enriched/` prevents re-processing

### Next Steps:

After downloading the results:
1. Upload `observations.geojson` to the Nuxt frontend's `public/data/` folder
2. Deploy to Netlify or serve statically
3. Or continue analysis in this notebook with the enriched CSVs